# Fine-Tuning Qwen2.5-Coder-7B-Instruct with QLoRA using Unsloth

This notebook fine-tunes `Qwen/Qwen2.5-Coder-7B-Instruct` on pull request code review datasets adhering to custom engineering standards.

### Key Highlights:
- **Framework**: Unsloth FastLanguageModel (2x faster, 70% less VRAM)
- **Quantization**: 4-bit NormalFloat (NF4) QLoRA
- **Rank**: $r=32$, $\alpha=64$, targeting all linear projections (`q, k, v, o, gate, up, down`)
- **Max Sequence Length**: 8,192 tokens
- **Packing**: `packing=False` (preserving variable-length PR diff structure)
- **Zero Contamination**: Strict assertion ensuring no dev/test repositories appear in training

In [ ]:
# 1. Check GPU environment
!nvidia-smi

In [ ]:
# 2. Install Unsloth and dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers pydantic pyyaml httpx

In [ ]:
# 3. Clone Repository or prepare dataset
!git clone https://github.com/example/codeagent.git /content/codeagent
%cd /content/codeagent
!python -m training.data_prep --clean-ratio 0.25 --distill

In [ ]:
# 4. Load Base Model with Unsloth
import torch
from unsloth import FastLanguageModel

max_seq_length = 8192
dtype = None  # Auto detection (Float16 or Bfloat16)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Add LoRA adapters targeting all linear layers
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# 5. Configure SFTTrainer
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

train_dataset = load_dataset("json", data_files="data/training/train_sft.jsonl", split="train")
dev_dataset = load_dataset("json", data_files="data/training/dev_sft.jsonl", split="train")

training_args = TrainingArguments(
    output_dir="models/qwen2.5-coder-7b-prism",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_ratio=0.05,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=50,
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    dataset_text_field="messages",
    max_seq_length=max_seq_length,
    packing=False,
    args=training_args,
)

# Start training
trainer_stats = trainer.train()

In [ ]:
# 6. Save LoRA Adapter & Merged 16-bit Weights
# Save LoRA adapter
model.save_pretrained_merged("models/qwen2.5-coder-7b-prism/adapter", tokenizer, save_method="lora")

# Save 16-bit merged weights for vLLM deployment
model.save_pretrained_merged(
    "models/qwen2.5-coder-7b-prism/merged_16bit", tokenizer, save_method="merged_16bit"
)

# Optional: Push to Hugging Face Hub
# model.push_to_hub_merged("username/Qwen2.5-Coder-7B-Instruct-Prism-Review", tokenizer, save_method="merged_16bit")